In [1]:
!pip install -q darts

In [2]:
import os, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from darts import TimeSeries
from darts.models import RNNModel, NBEATSModel, TCNModel, TransformerModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.missing_values import fill_missing_values
from darts.metrics import mae, rmse, smape

warnings.filterwarnings('ignore')
torch.manual_seed(0)
np.random.seed(0)

print('GPU available:', torch.cuda.is_available())

GPU available: True


In [3]:
QUICK_MODE = True
DATASET_LIST = [
    ('daily_waste_output_Boralasgamuwa', '2016-05-01'),
    ('daily_waste_output_Moratuwa', '2017-11-01'),
    ('daily_waste_output_Dehiwala', '2015-02-01'),
    ('daily_waste_output_Ballarat', '2008-09-01'),
    ('daily_waste_output_Austin', '20l15-06-01'),
]

DATA_DIR = '/content/data'
DATE_COL = 'date'
VALUE_COL = 'waste'

COLUMN_OVERRIDES = {}
if QUICK_MODE:
    INPUT_LENGTHS = [7, 30]
    EPOCHS = {'LSTM': 10, 'N-BEATS': 10, 'TCN': 10, 'Transformer': 10}
else:
    INPUT_LENGTHS = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 30, 60, 90, 120, 150,
                     180, 210, 240, 270, 300, 330, 360]
    EPOCHS = {'LSTM': 400, 'N-BEATS': 200, 'TCN': 500, 'Transformer': 200}

MODELS_TO_RUN = ['LSTM', 'N-BEATS', 'TCN', 'Transformer']

LEARNING_RATE = 1e-4
BATCH_SIZE = 32

In [11]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
os.makedirs(DATA_DIR, exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, os.path.join(DATA_DIR, fname))
except ImportError:
    print('Not running on Colab - put your CSV files in', DATA_DIR, 'manually.')

print('Files in data folder:', os.listdir(DATA_DIR))

Saving daily_waste_output_Austin.csv to daily_waste_output_Austin.csv
Saving daily_waste_output_Ballarat.csv to daily_waste_output_Ballarat.csv
Saving daily_waste_output_Boralasgamuwa.csv to daily_waste_output_Boralasgamuwa.csv
Saving daily_waste_output_Dehiwala.csv to daily_waste_output_Dehiwala.csv
Saving daily_waste_output_Moratuwa.csv to daily_waste_output_Moratuwa.csv
Files in data folder: ['daily_waste_output_Moratuwa.csv', 'daily_waste_output_Boralasgamuwa.csv', 'daily_waste_output_Dehiwala.csv', 'daily_waste_output_Ballarat.csv', 'daily_waste_output_Austin.csv']


In [6]:
for name, _ in DATASET_LIST:
    path = os.path.join(DATA_DIR, name + '.csv')
    if os.path.exists(path):
        print('==', name)
        display(pd.read_csv(path).head(3))

== daily_waste_output_Boralasgamuwa


,ticket_date,net_weight_kg
0,2012-01-01 00:00:00+00:00,17340.0
1,2012-01-02 00:00:00+00:00,25980.0
2,2012-01-03 00:00:00+00:00,29220.0


== daily_waste_output_Moratuwa


,ticket_date,net_weight_kg
0,2015-01-01 00:00:00+00:00,63550.0
1,2015-01-02 00:00:00+00:00,94850.0
2,2015-01-03 00:00:00+00:00,96180.0


== daily_waste_output_Dehiwala


,ticket_date,net_weight_kg
0,2012-01-01 00:00:00+00:00,82270.0
1,2012-01-02 00:00:00+00:00,123010.0
2,2012-01-03 00:00:00+00:00,128330.0


== daily_waste_output_Ballarat


,ticket_date,net_weight_kg
0,2001-01-01 00:00:00+00:00,61887.5
1,2001-01-02 00:00:00+00:00,61620.0
2,2001-01-03 00:00:00+00:00,71040.0


== daily_waste_output_Austin


,ticket_date,net_weight_kg
0,2001-01-01 00:00:00+00:00,-1500000.0
1,2001-01-02 00:00:00+00:00,-1500000.0
2,2001-01-03 00:00:00+00:00,-1500000.0


## 5. Helper functions

In [7]:
def load_series(dataset_name):
    # Read the CSV and turn it into a clean DAILY Darts TimeSeries.
    date_col, value_col = COLUMN_OVERRIDES.get(dataset_name, (DATE_COL, VALUE_COL))
    path = os.path.join(DATA_DIR, dataset_name + '.csv')

    df = pd.read_csv(path)
    df[date_col] = pd.to_datetime(df[date_col])
    # If a day appears several times (e.g. several trucks), add them up.
    df = df.groupby(date_col, as_index=False)[value_col].sum()

    series = TimeSeries.from_dataframe(
        df, time_col=date_col, value_cols=value_col,
        freq='D', fill_missing_dates=True)          # add rows for missing days (as NaN)
    series = fill_missing_values(series)            # fill those NaNs by interpolation
    return series


def score(actual, pred):
    return {
        'MAE': mae(actual, pred),       # average absolute error (same unit as waste)
        'RMSE': rmse(actual, pred),     # like MAE but punishes big mistakes more
        'SMAPE': smape(actual, pred),   # percentage error (0-200 %)
    }

In [8]:
def build_model(name, input_len, train_len):

    trainer = {'accelerator': 'auto',            # GPU if available, else CPU
               'enable_progress_bar': False,
               'enable_model_summary': False}
    common = dict(n_epochs=EPOCHS[name],
                  batch_size=BATCH_SIZE,
                  optimizer_kwargs={'lr': LEARNING_RATE},
                  random_state=0,                # same result every run
                  force_reset=True,
                  pl_trainer_kwargs=trainer)

    if name == 'LSTM':
        training_length = max(330, input_len + 30)
        training_length = min(training_length, train_len - 1)
        if training_length <= input_len:
            return None
        return RNNModel(model='LSTM',
                        input_chunk_length=input_len,
                        training_length=training_length,
                        hidden_dim=32, n_rnn_layers=1, dropout=0.2,
                        **common)

    if name == 'N-BEATS':
        return NBEATSModel(input_chunk_length=input_len, output_chunk_length=1,
                           num_stacks=10, num_blocks=1, num_layers=4,
                           layer_widths=512, generic_architecture=True,
                           **common)

    if name == 'TCN':
        kernel_size = 3
        if input_len <= kernel_size:             # TCN needs input_len > kernel_size
            return None
        return TCNModel(input_chunk_length=input_len, output_chunk_length=1,
                        kernel_size=kernel_size, dilation_base=2,
                        num_filters=10, dropout=0.2,
                        **common)

    if name == 'Transformer':
        return TransformerModel(input_chunk_length=input_len, output_chunk_length=1,
                                d_model=16, nhead=8,
                                num_encoder_layers=2, num_decoder_layers=2,
                                dim_feedforward=128, dropout=0.2,
                                **common)

    raise ValueError('Unknown model: ' + name)

In [9]:
def run_one(dataset_name, test_start, model_name, input_len, series, train, test):
    scaler = Scaler()
    train_s = scaler.fit_transform(train)
    series_s = scaler.transform(series)
    model = build_model(model_name, input_len, len(train))
    if model is None:
        return None, 'skipped (window size not valid for this model)'
    model.fit(train_s)


    pred_s = model.historical_forecasts(
        series_s, start=test.start_time(),
        forecast_horizon=1, stride=1,
        retrain=False, last_points_only=True, verbose=False)

    pred = scaler.inverse_transform(pred_s)
    actual = series.slice_intersect(pred)
    return (pred, score(actual, pred)), 'ok'

In [13]:
results = []
predictions = {}

for dataset_name, test_start in DATASET_LIST:
    path = os.path.join(DATA_DIR, dataset_name + '.csv')
    if not os.path.exists(path):
        print(f'[skip] {dataset_name}: CSV not found in {DATA_DIR}')
        continue

    series = load_series(dataset_name)
    train, test = series.split_before(pd.Timestamp(test_start))
    print(f'\n=== {dataset_name} | train days: {len(train)} | test days: {len(test)}')

    # ---- naive baselines: "same as yesterday" and "same as 7 days ago"
    for label, lag in [('Naive (yesterday)', 1), ('Naive (last week)', 7)]:
        pred = series.shift(lag).slice_intersect(test)
        actual = series.slice_intersect(pred)
        results.append({'dataset': dataset_name, 'model': label, 'input_len': 0,
                        **score(actual, pred), 'seconds': 0, 'status': 'ok'})
        predictions[(dataset_name, label, 0)] = pred

    # ---- deep learning models
    for model_name in MODELS_TO_RUN:
        for input_len in INPUT_LENGTHS:
            t0 = time.time()
            try:
                out, status = run_one(dataset_name, test_start, model_name,
                                      input_len, series, train, test)
            except Exception as e:                   # we KEEP the error message
                out, status = None, f'error: {type(e).__name__}: {str(e)[:120]}'
            secs = round(time.time() - t0, 1)

            row = {'dataset': dataset_name, 'model': model_name,
                   'input_len': input_len, 'seconds': secs, 'status': status}
            if out is not None:A
                pred, metrics = out
                row.update(metrics)
                predictions[(dataset_name, model_name, input_len)] = pred
                print(f'{model_name:12s} inp={input_len:<4d} RMSE={metrics["RMSE"]:.2f}  ({secs}s)')
            else:
                print(f'{model_name:12s} inp={input_len:<4d} {status}')
            results.append(row)

results_df = pd.DataFrame(results)
results_df.to_csv('results.csv', index=False)
print('\nDone. Saved to results.csv')

In [ ]:
ok = results_df[results_df['status'] == 'ok'].copy()

print('Top 5 per dataset (lowest RMSE):')
top = ok.sort_values('RMSE').groupby('dataset').head(5).sort_values(['dataset', 'RMSE'])
display(top[['dataset', 'model', 'input_len', 'MAE', 'RMSE', 'SMAPE', 'seconds']].round(2))

print('\nBest window size per model:')
best_per_model = ok.loc[ok.groupby(['dataset', 'model'])['RMSE'].idxmin()]
display(best_per_model[['dataset', 'model', 'input_len', 'MAE', 'RMSE', 'SMAPE']].round(2))

failed = results_df[~results_df['status'].isin(['ok'])]
print(f'\nRuns skipped/failed: {len(failed)}')
if len(failed):
    display(failed[['dataset', 'model', 'input_len', 'status']].head(20))

In [ ]:

pivot = best_per_model.pivot(index='dataset', columns='model', values='RMSE')
pivot.plot(kind='bar', figsize=(11, 4))
plt.ylabel('RMSE (lower is better)')
plt.title('Best RMSE per model')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Plot real vs predicted for the best deep-learning model of each dataset (last 120 test days)
for dataset_name, test_start in DATASET_LIST:
    sub = ok[(ok['dataset'] == dataset_name) & (~ok['model'].str.startswith('Naive'))]
    if sub.empty:
        continue
    best = sub.sort_values('RMSE').iloc[0]
    pred = predictions[(dataset_name, best['model'], best['input_len'])]
    series = load_series(dataset_name)
    actual = series.slice_intersect(pred)

    plt.figure(figsize=(12, 3.5))
    actual[-120:].plot(label='Actual')
    pred[-120:].plot(label=f"{best['model']} (inp={best['input_len']})")
    plt.title(dataset_name)
    plt.legend()
    plt.show()